# Cognitive Task PCA and EFA

Run matched cognitive-measure PCA and exploratory factor analysis workflows, both without SES residualization and with INR included in the residualization model.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import inspect
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy import stats
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo
import factor_analyzer.factor_analyzer as factor_analyzer_module
from sklearn.linear_model import LinearRegression
import seaborn as sns


# factor_analyzer versions that call sklearn's older force_all_finite argument need
# this compatibility layer with newer sklearn releases that renamed it.
if 'force_all_finite' not in inspect.signature(factor_analyzer_module.check_array).parameters:
    _sklearn_check_array = factor_analyzer_module.check_array

    def _factor_analyzer_check_array(*args, force_all_finite=True, **kwargs):
        kwargs.setdefault('ensure_all_finite', force_all_finite)
        return _sklearn_check_array(*args, **kwargs)

    factor_analyzer_module.check_array = _factor_analyzer_check_array


sns.set_style('whitegrid')
sns.set_palette('colorblind')
palette = sns.color_palette('colorblind')
plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 26,
    'axes.labelsize': 22,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
})


In [ ]:
INVALID_CODES = (555, 777, 888, 999)

DEFAULT_CATEGORICAL_COVARIATES = {
    'demo_sex_v2',
    'site_id_l',
    'ehi1b',
}

def _validate_columns(df, columns, context):
    missing = [column for column in columns if column not in df.columns]
    if missing:
        raise KeyError(f'Missing required {context} column(s): {missing}')


def standardize_subject_id(subject_ids):
    return (
        subject_ids.astype(str)
        .str.replace('_', '', regex=False)
        .str.replace('^sub-', '', regex=True)
    )


def load_motion_qa(motion_qa_path, motion_column='mean_fd_0.20'):
    motion_qa = pd.read_csv(motion_qa_path)
    _validate_columns(motion_qa, ['src_subject_id', motion_column], 'motion QA')
    motion_qa = motion_qa[['src_subject_id', motion_column]].copy()
    motion_qa['src_subject_id'] = standardize_subject_id(motion_qa['src_subject_id'])
    motion_qa = motion_qa.drop_duplicates(subset=['src_subject_id'])
    motion_qa[motion_column] = pd.to_numeric(motion_qa[motion_column], errors='coerce')
    return motion_qa


def merge_motion_qa(df, motion_qa_path, motion_column='mean_fd_0.20', how='left'):
    _validate_columns(df, ['src_subject_id'], 'analysis')
    motion_qa = load_motion_qa(motion_qa_path, motion_column=motion_column)

    merged = df.copy()
    merged['src_subject_id'] = standardize_subject_id(merged['src_subject_id'])
    if motion_column in merged.columns:
        existing_motion = merged.groupby('src_subject_id')[motion_column].first()
        merged = merged.drop(columns=[motion_column])
        merged = merged.merge(motion_qa, on='src_subject_id', how=how)
        merged[motion_column] = merged[motion_column].fillna(
            merged['src_subject_id'].map(existing_motion)
        )
    else:
        merged = merged.merge(motion_qa, on='src_subject_id', how=how)
    return merged


def _encode_regression_covariates(df, covariates, categorical_covariates=DEFAULT_CATEGORICAL_COVARIATES):
    encoded_parts = []
    categorical_covariates = set(categorical_covariates)

    for covariate in covariates:
        if df[covariate].dtype == 'object' or covariate in categorical_covariates:
            encoded_parts.append(
                pd.get_dummies(
                    df[covariate],
                    prefix=covariate,
                    drop_first=True,
                    dtype=float,
                )
            )
        else:
            encoded_parts.append(
                pd.to_numeric(df[covariate], errors='coerce').to_frame(covariate)
            )

    if not encoded_parts:
        return pd.DataFrame(index=df.index)
    return pd.concat(encoded_parts, axis=1)


def residualize_measure(df, measure, covariates, output_col=None, invalid_codes=INVALID_CODES):
    _validate_columns(df, ['src_subject_id', measure, *covariates], 'residualization')
    df = df.copy()
    output_col = output_col or f'{measure}_resid'
    if output_col in df.columns:
        df = df.drop(columns=[output_col])

    temp_df = df[['src_subject_id', measure, *covariates]].copy()
    temp_df[measure] = temp_df[measure].replace(list(invalid_codes), np.nan)
    temp_df = temp_df.dropna()

    if len(temp_df) == 0:
        df[output_col] = np.nan
        return df

    covariate_matrix = _encode_regression_covariates(temp_df, covariates)
    if covariate_matrix.shape[1] == 0:
        predicted = np.repeat(temp_df[measure].mean(), len(temp_df))
    else:
        model = LinearRegression().fit(covariate_matrix, temp_df[measure])
        predicted = model.predict(covariate_matrix)
    temp_df[output_col] = temp_df[measure] - predicted

    return df.merge(
        temp_df[['src_subject_id', output_col]],
        on='src_subject_id',
        how='left',
    )


In [ ]:
DATA_PATH = '/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability/Final/wrangled_pMTG_FC_data_midb61_meanFC.csv'
MOTION_QA_PATH = '/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability/Final/motion_QA_results.csv'
PCA_RESULTS_DIR = Path('/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability/Final/pca_results')
PCA_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df = merge_motion_qa(df, MOTION_QA_PATH, how='left')
missing_motion = df['mean_fd_0.20'].isna().sum() if 'mean_fd_0.20' in df.columns else len(df)
print(f'Loaded {len(df)} rows from {DATA_PATH}')
print(f'Missing mean_fd_0.20 values after motion QA merge: {missing_motion}')


## Task Analysis Settings

The SES model residualizes raw cognitive measures once with INR included alongside the standard age and sex covariates.

In [ ]:
SES_VARIABLE = 'inr'
SES_RESIDUALIZATION_COVARIATE = 'inr'
INVALID_CODES = (555, 777, 888, 999)
EFA_N_FACTORS = None  # None uses the Kaiser rule: retain factors with eigenvalue > 1.
EFA_ROTATION = 'oblimin'
EFA_METHOD = 'ml'

COGNITIVE_TASK_MEASURES = [
    'ravlt_immediate_2y',
    'ravlt_short_delay_2y',
    'ravlt_long_delay_2y',
    'nihtbx_picvocab_uncorrected_2y',
    'nihtbx_reading_uncorrected_2y',
    'nihtbx_picture_uncorrected_2y',
    'nihtbx_flanker_uncorrected_2y'
]

column_names_to_tasks = {
    'ravlt_immediate_2y': 'RAVLT Immediate Recall',
    'ravlt_short_delay_2y': 'RAVLT Short Delayed Recall',
    'ravlt_long_delay_2y': 'RAVLT Long Delayed Recall',
    'nihtbx_picvocab_uncorrected_2y': 'Picture Vocabulary',
    'nihtbx_reading_uncorrected_2y': 'Oral Reading Recognition',
    'nihtbx_picture_uncorrected_2y': 'Picture Sequence Memory',
    'nihtbx_flanker_uncorrected_2y': 'Flanker'
}

missing_cognitive_cols = [
    measure for measure in COGNITIVE_TASK_MEASURES if measure not in df.columns
]
if missing_cognitive_cols:
    raise KeyError(f'Missing cognitive task columns for PCA/EFA: {missing_cognitive_cols}')

required_inr_cols = [SES_VARIABLE, 'inr_missing', 'poverty_line_2017']
missing_inr_cols = [col for col in required_inr_cols if col not in df.columns]
if missing_inr_cols:
    raise KeyError(
        'Missing INR columns from the wrangled data table: '
        f'{missing_inr_cols}. Run data_wrangling.ipynb before PCA_tasks.ipynb.'
    )

COGNITIVE_TASK_COVARIATES = ['interview_age', 'demo_sex_v2', 'site_id_l', 'ehi1b']
PCA_MODEL_SPECS = [
    {
        'model_name': 'without_ses_residualization',
        'label': 'No SES residualization',
        'short_label': 'No SES',
        'prefix': 'no_ses',
        'cognitive_covariates': COGNITIVE_TASK_COVARIATES,
    },
    {
        'model_name': 'with_ses_residualization',
        'label': 'SES residualization',
        'short_label': 'SES',
        'prefix': 'ses',
        'cognitive_covariates': COGNITIVE_TASK_COVARIATES + [SES_RESIDUALIZATION_COVARIATE],
    },
]

for spec in PCA_MODEL_SPECS:
    missing_cognitive_covariates = [
        covariate for covariate in spec['cognitive_covariates'] if covariate not in df.columns
    ]
    if missing_cognitive_covariates:
        raise KeyError(
            f"Missing cognitive residualization covariates for {spec['model_name']}: "
            f'{missing_cognitive_covariates}'
        )

print(f'Missing raw INR values: {df[SES_VARIABLE].isna().sum()}')
print('Task PCA/EFA model specs:')
for spec in PCA_MODEL_SPECS:
    print(f"- {spec['model_name']}")
    print(f"  Cognitive covariates: {spec['cognitive_covariates']}")
print(f'EFA factor count: {EFA_N_FACTORS if EFA_N_FACTORS is not None else "Kaiser eigenvalue > 1 rule"}')
print(f'EFA method: {EFA_METHOD}; rotation: {EFA_ROTATION}')


## Shared Task PCA/EFA Helpers

In [ ]:
def residualize_cognitive_variant(data, measures, covariates, output_suffix):
    """Residualize raw cognitive measures once using the full covariate set."""
    residualized = data
    output_cols = []
    for measure in measures:
        output_col = f'{measure}_{output_suffix}_resid'
        residualized = residualize_measure(
            residualized,
            measure=measure,
            covariates=covariates,
            output_col=output_col,
            invalid_codes=INVALID_CODES,
        )
        output_cols.append(output_col)
    return residualized, output_cols


def fit_pca_scores(data, feature_cols, score_prefix, scale_features=True):
    """Fit PCA and return full-index score, variance, and loading tables."""
    pca_input = data[feature_cols].apply(pd.to_numeric, errors='coerce')
    pca_complete = pca_input.dropna().copy()
    print(f'{score_prefix} PCA complete cases: {len(pca_complete)}/{len(pca_input)}')
    if len(pca_complete) < 2:
        raise ValueError(f'Not enough complete rows to run {score_prefix} PCA.')

    if scale_features:
        pca_matrix = StandardScaler().fit_transform(pca_complete)
    else:
        pca_matrix = (pca_complete - pca_complete.mean()).to_numpy()

    pca = PCA(random_state=42)
    scores_array = pca.fit_transform(pca_matrix)
    score_cols = [f'{score_prefix}_PC{i + 1}' for i in range(scores_array.shape[1])]

    scores = pd.DataFrame(np.nan, index=data.index, columns=score_cols, dtype=float)
    scores.loc[pca_complete.index, score_cols] = scores_array

    variance = pd.DataFrame({
        'component': score_cols,
        'explained_variance_ratio': pca.explained_variance_ratio_,
        'cumulative_explained_variance_ratio': np.cumsum(pca.explained_variance_ratio_),
    })

    loadings = pd.DataFrame(
        pca.components_.T * np.sqrt(pca.explained_variance_),
        index=feature_cols,
        columns=score_cols,
    )

    return {
        'input': pca_input,
        'complete_input': pca_complete,
        'pca': pca,
        'score_cols': score_cols,
        'scores': scores,
        'variance': variance,
        'loadings': loadings,
    }


def attach_score_columns(data, scores):
    updated = data.copy()
    for score_col in scores.columns:
        updated[score_col] = scores[score_col]
    return updated


def factorability_diagnostics(efa_complete):
    """Return KMO and Bartlett diagnostics for complete EFA input rows."""
    diagnostics = {}
    try:
        bartlett_chi_square, bartlett_p_value = calculate_bartlett_sphericity(efa_complete)
    except Exception as exc:
        print(f'Bartlett sphericity test failed: {exc}')
        bartlett_chi_square, bartlett_p_value = np.nan, np.nan
    try:
        kmo_per_item, kmo_overall = calculate_kmo(efa_complete)
    except Exception as exc:
        print(f'KMO calculation failed: {exc}')
        kmo_per_item = np.full(efa_complete.shape[1], np.nan)
        kmo_overall = np.nan

    diagnostics['bartlett_chi_square'] = bartlett_chi_square
    diagnostics['bartlett_p_value'] = bartlett_p_value
    diagnostics['kmo_overall'] = kmo_overall
    diagnostics['kmo_per_item'] = pd.Series(kmo_per_item, index=efa_complete.columns, name='kmo')
    return diagnostics


def choose_efa_factor_count(eigenvalues, requested_n_factors=None):
    """Choose a non-saturated EFA factor count, defaulting to eigenvalues > 1."""
    max_factors = max(1, len(eigenvalues) - 1)
    if requested_n_factors is None:
        selected = int(np.sum(np.asarray(eigenvalues) > 1.0))
        selection_rule = 'kaiser_eigenvalue_gt_1'
        if selected < 1:
            selected = 1
            selection_rule = 'kaiser_fallback_1_factor'
    else:
        selected = int(requested_n_factors)
        selection_rule = 'user_specified'

    selected = min(max(selected, 1), max_factors)
    return selected, selection_rule


def fit_efa_scores(
    data,
    feature_cols,
    score_prefix,
    n_factors=None,
    rotation='oblimin',
    method='ml',
):
    """Fit EFA and return full-index factor score, loading, variance, and diagnostic tables."""
    efa_input = data[feature_cols].apply(pd.to_numeric, errors='coerce')
    efa_complete = efa_input.dropna().copy()
    print(f'{score_prefix} EFA complete cases: {len(efa_complete)}/{len(efa_input)}')
    if len(efa_complete) < 2:
        raise ValueError(f'Not enough complete rows to run {score_prefix} EFA.')
    if len(feature_cols) < 2:
        raise ValueError(f'Need at least two measures to run {score_prefix} EFA.')

    scaler = StandardScaler()
    efa_matrix = pd.DataFrame(
        scaler.fit_transform(efa_complete),
        index=efa_complete.index,
        columns=feature_cols,
    )

    eigen_model = FactorAnalyzer(rotation=None, method=method, svd_method='lapack')
    eigen_model.fit(efa_matrix)
    eigenvalues, common_factor_eigenvalues = eigen_model.get_eigenvalues()
    selected_n_factors, selection_rule = choose_efa_factor_count(eigenvalues, n_factors)

    efa = FactorAnalyzer(
        n_factors=selected_n_factors,
        rotation=rotation,
        method=method,
        svd_method='lapack',
    )
    efa.fit(efa_matrix)
    factor_score_array = efa.transform(efa_matrix)
    factor_cols = [f'{score_prefix}_EF{i + 1}' for i in range(selected_n_factors)]

    scores = pd.DataFrame(np.nan, index=data.index, columns=factor_cols, dtype=float)
    scores.loc[efa_complete.index, factor_cols] = factor_score_array

    loadings = pd.DataFrame(efa.loadings_, index=feature_cols, columns=factor_cols)
    uniquenesses = pd.Series(efa.get_uniquenesses(), index=feature_cols, name='uniqueness')
    communalities = pd.Series(efa.get_communalities(), index=feature_cols, name='communality')
    measure_diagnostics = pd.concat([communalities, uniquenesses], axis=1)

    variance_values, proportional_variance, cumulative_variance = efa.get_factor_variance()
    variance = pd.DataFrame({
        'factor': factor_cols,
        'ss_loadings': variance_values,
        'proportional_variance': proportional_variance,
        'cumulative_variance': cumulative_variance,
    })

    eigenvalues_table = pd.DataFrame({
        'component': np.arange(1, len(eigenvalues) + 1),
        'eigenvalue': eigenvalues,
        'common_factor_eigenvalue': common_factor_eigenvalues,
        'retained_by_kaiser_rule': eigenvalues > 1.0,
    })

    factorability = factorability_diagnostics(efa_complete)
    diagnostics = pd.DataFrame([
        {
            'score_prefix': score_prefix,
            'n_complete': len(efa_complete),
            'n_measures': len(feature_cols),
            'n_factors': selected_n_factors,
            'factor_selection_rule': selection_rule,
            'rotation': rotation,
            'method': method,
            'bartlett_chi_square': factorability['bartlett_chi_square'],
            'bartlett_p_value': factorability['bartlett_p_value'],
            'kmo_overall': factorability['kmo_overall'],
        }
    ])
    kmo_per_item = factorability['kmo_per_item'].to_frame()

    return {
        'input': efa_input,
        'complete_input': efa_complete,
        'standardized_input': efa_matrix,
        'efa': efa,
        'score_cols': factor_cols,
        'scores': scores,
        'loadings': loadings,
        'variance': variance,
        'eigenvalues': eigenvalues_table,
        'diagnostics': diagnostics,
        'measure_diagnostics': measure_diagnostics,
        'kmo_per_item': kmo_per_item,
    }


def plot_cognitive_efa_summary(efa_result, labels, title):
    """Display EFA eigenvalues and rotated loadings."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    eigenvalues = efa_result['eigenvalues']
    axes[0].plot(
        eigenvalues['component'],
        eigenvalues['eigenvalue'],
        marker='o',
        color='black',
    )
    axes[0].axhline(1.0, color=palette[1], linestyle='--', linewidth=1.0, label='Eigenvalue = 1')
    axes[0].set_xlabel('Factor Number', fontsize=16)
    axes[0].set_ylabel('Eigenvalue', fontsize=16)
    axes[0].set_title(title, fontsize=20)
    axes[0].set_xticks(eigenvalues['component'])
    axes[0].legend(frameon=False)

    heatmap = sns.heatmap(
        efa_result['loadings'],
        ax=axes[1],
        center=0,
        cmap='vlag',
        annot=True,
        fmt='.2f',
        cbar_kws={'label': 'Rotated Loading'},
    )

    # Set colorbar label font size separately
    colorbar = heatmap.collections[0].colorbar
    colorbar.ax.yaxis.label.set_size(14)
    colorbar.ax.tick_params(labelsize=12)

    axes[1].set_title('Factor Loadings for Latent Retrieval Variables', fontsize=16)
    axes[1].set_xlabel('Exploratory Factor', fontsize=14)
    axes[1].set_xticks(np.arange(len(efa_result['score_cols'])) + 0.5)
    axes[1].set_xticklabels(
        [col.split('_')[-1] for col in efa_result['score_cols']],
        rotation=0,
        fontsize=14,
    )
    axes[1].set_yticks(np.arange(len(labels)) + 0.5)
    axes[1].set_yticklabels(labels, rotation=0, fontsize=14)
    axes[1].set_ylabel('Cognitive Task Measure', fontsize=14)
    plt.tight_layout()
    plt.show()
    plt.close()


def task_labels_for_residual_cols(cols, output_suffix):
    suffix = f'_{output_suffix}_resid'
    return [
        column_names_to_tasks.get(col.removesuffix(suffix), col)
        for col in cols
    ]


def plot_cognitive_measure_correlations(pca_input, feature_cols, labels, title):
    correlations = pca_input[feature_cols].corr(method='pearson')
    pairwise_n = pca_input[feature_cols].notna().astype(int).T.dot(
        pca_input[feature_cols].notna().astype(int)
    )

    print(f'Pearson correlations: {title}')
    display(correlations)
    print('Pairwise non-missing sample sizes:')
    display(pairwise_n)

    plt.figure(figsize=(11, 9))
    sns.heatmap(
        correlations,
        vmin=-1,
        vmax=1,
        center=0,
        cmap='vlag',
        annot=True,
        fmt='.2f',
        square=True,
        cbar_kws={'label': 'Pearson r'},
    )
    plt.title(title, fontsize=20)
    plt.xticks(
        ticks=np.arange(len(feature_cols)) + 0.5,
        labels=labels,
        rotation=45,
        ha='right',
        fontsize=16,
    )
    plt.yticks(
        ticks=np.arange(len(feature_cols)) + 0.5,
        labels=labels,
        rotation=0,
        fontsize=16,
    )
    plt.tight_layout()
    plt.show()
    plt.close()

    return correlations, pairwise_n


def plot_cognitive_pca_summary(pca_result, labels, title):
    n_components_to_plot = min(10, len(pca_result['score_cols']))

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    axes[0].plot(
        np.arange(1, len(pca_result['score_cols']) + 1),
        pca_result['variance']['explained_variance_ratio'],
        marker='o',
        color='black',
        label='Individual',
    )
    axes[0].plot(
        np.arange(1, len(pca_result['score_cols']) + 1),
        pca_result['variance']['cumulative_explained_variance_ratio'],
        marker='o',
        color=palette[0],
        label='Cumulative',
    )
    axes[0].set_xlabel('Principal Component', fontsize=14)
    axes[0].set_ylabel('Variance Explained', fontsize=14)
    axes[0].set_title(title, fontsize=20)
    axes[0].set_xticks(np.arange(1, len(pca_result['score_cols']) + 1))
    axes[0].legend(frameon=False)

    sns.heatmap(
        pca_result['loadings'].iloc[:, :n_components_to_plot],
        ax=axes[1],
        center=0,
        cmap='vlag',
        annot=True,
        fmt='.2f',
        cbar_kws={'label': 'Loading'},
    )
    axes[1].set_title('Cognitive PCA Loadings', fontsize=16)
    axes[1].set_xlabel('Principal Component', fontsize=14)
    pc_labels = [col.split('_')[-1] for col in pca_result['score_cols'][:n_components_to_plot]]
    axes[1].set_xticks(np.arange(n_components_to_plot) + 0.5)
    axes[1].set_xticklabels(pc_labels, rotation=0, fontsize=14)
    axes[1].set_yticks(np.arange(len(labels)) + 0.5)
    axes[1].set_yticklabels(labels, rotation=0, fontsize=14)
    axes[1].set_ylabel('Cognitive Task Measure')
    plt.tight_layout()
    plt.show()
    plt.close()

def plot_residual_normality_diagnostics(data, residual_cols, labels=None, title_prefix='Residual Normality', page_size=8, bins=30):
    """Display histogram and Q-Q plots for residual columns, plus summary moments."""
    residual_data = data[residual_cols].apply(pd.to_numeric, errors='coerce')
    labels = labels or residual_cols
    if len(labels) != len(residual_cols):
        raise ValueError('labels must have the same length as residual_cols')

    summary_rows = []
    for col, label in zip(residual_cols, labels):
        values = residual_data[col].dropna()
        summary_rows.append({
            'residual': col,
            'label': label,
            'n': len(values),
            'mean': values.mean(),
            'sd': values.std(ddof=1),
            'skew': stats.skew(values) if len(values) >= 3 else np.nan,
            'excess_kurtosis': stats.kurtosis(values) if len(values) >= 4 else np.nan,
        })

    summary = pd.DataFrame(summary_rows)
    print(f'{title_prefix}: residual normality summary')
    display(summary)

    for start in range(0, len(residual_cols), page_size):
        page_cols = residual_cols[start:start + page_size]
        page_labels = labels[start:start + page_size]
        fig, axes = plt.subplots(len(page_cols), 2, figsize=(13, 3.2 * len(page_cols)))
        axes = np.asarray(axes).reshape(len(page_cols), 2)

        for row_idx, (col, label) in enumerate(zip(page_cols, page_labels)):
            values = residual_data[col].dropna()
            hist_ax, qq_ax = axes[row_idx]

            if values.empty:
                hist_ax.text(0.5, 0.5, 'No non-missing values', ha='center', va='center')
                qq_ax.text(0.5, 0.5, 'No non-missing values', ha='center', va='center')
            else:
                sns.histplot(values, bins=bins, kde=True, ax=hist_ax, color=palette[0])
                hist_ax.axvline(values.mean(), color='black', linewidth=1.0, linestyle='--')
                hist_ax.set_xlabel('Residual')
                hist_ax.set_ylabel('Count')

                if len(values) >= 2:
                    stats.probplot(values, dist='norm', plot=qq_ax)
                    qq_ax.get_lines()[0].set_markerfacecolor(palette[1])
                    qq_ax.get_lines()[0].set_markeredgecolor(palette[1])
                    qq_ax.get_lines()[1].set_color('black')
                    qq_ax.get_lines()[1].set_linewidth(1.0)
                else:
                    qq_ax.text(0.5, 0.5, 'Need at least 2 values', ha='center', va='center')

            hist_ax.set_title(f'{label} histogram', fontsize=13)
            qq_ax.set_title(f'{label} Q-Q plot', fontsize=13)

        page_number = start // page_size + 1
        fig.suptitle(f'{title_prefix} residual normality diagnostics, page {page_number}', fontsize=18, y=1.01)
        plt.tight_layout()
        plt.show()
        plt.close()

    return summary


## Matched Task PCA/EFA Workflows

In [ ]:
pca_model_outputs = {}

for spec in PCA_MODEL_SPECS:
    print('\n' + '=' * 90)
    print(spec['label'])
    print('=' * 90)
    print('Cognitive covariates:', spec['cognitive_covariates'])

    df, cognitive_resid_cols = residualize_cognitive_variant(
        df,
        COGNITIVE_TASK_MEASURES,
        covariates=spec['cognitive_covariates'],
        output_suffix=spec['prefix'],
    )
    task_labels = task_labels_for_residual_cols(cognitive_resid_cols, spec['prefix'])
    cognitive_residual_normality = plot_residual_normality_diagnostics(
        df,
        cognitive_resid_cols,
        labels=task_labels,
        title_prefix=f"{spec['short_label']} cognitive residuals",
        page_size=4,
    )

    cognitive_result = fit_pca_scores(
        df,
        cognitive_resid_cols,
        score_prefix=f"{spec['prefix']}_cognitive",
        scale_features=True,
    )
    df = attach_score_columns(df, cognitive_result['scores'])

    print('\nCognitive PCA variance explained:')
    display(cognitive_result['variance'])
    print('\nCognitive PCA loadings:')
    display(cognitive_result['loadings'])

    plot_cognitive_measure_correlations(
        cognitive_result['input'],
        cognitive_resid_cols,
        task_labels,
        title=f"Correlations Between Cognitive Tasks",
    )
    plot_cognitive_pca_summary(
        cognitive_result,
        task_labels,
        title=f"Cognitive Task PCA",
    )

    cognitive_efa_result = fit_efa_scores(
        df,
        cognitive_resid_cols,
        score_prefix=f"{spec['prefix']}_cognitive",
        n_factors=EFA_N_FACTORS,
        rotation=EFA_ROTATION,
        method=EFA_METHOD,
    )
    df = attach_score_columns(df, cognitive_efa_result['scores'])

    print('\nCognitive EFA diagnostics:')
    display(cognitive_efa_result['diagnostics'])
    print('\nCognitive EFA eigenvalues:')
    display(cognitive_efa_result['eigenvalues'])
    print('\nCognitive EFA variance:')
    display(cognitive_efa_result['variance'])
    print('\nCognitive EFA rotated loadings:')
    display(cognitive_efa_result['loadings'])
    print('\nCognitive EFA measure diagnostics:')
    display(cognitive_efa_result['measure_diagnostics'].join(cognitive_efa_result['kmo_per_item']))

    plot_cognitive_efa_summary(
        cognitive_efa_result,
        task_labels,
        title=f"{spec['short_label']} Cognitive Task EFA",
    )

    pca_model_outputs[spec['model_name']] = {
        'spec': spec,
        'cognitive_resid_cols': cognitive_resid_cols,
        'task_labels': task_labels,
        'cognitive_residual_normality': cognitive_residual_normality,
        'cognitive': cognitive_result,
        'cognitive_efa': cognitive_efa_result,
    }

no_ses_model = pca_model_outputs['without_ses_residualization']
ses_model = pca_model_outputs['with_ses_residualization']


## Export Task PCA/EFA Results

In [ ]:
PCA_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
pca_export_manifest_rows = []

for model_name, model_output in pca_model_outputs.items():
    cognitive_output = model_output['cognitive']
    cognitive_efa_output = model_output['cognitive_efa']

    pca_scores_path = PCA_RESULTS_DIR / f'pca_cognitive_scores_{model_name}.csv'
    pca_variance_path = PCA_RESULTS_DIR / f'pca_cognitive_variance_{model_name}.csv'
    pca_loadings_path = PCA_RESULTS_DIR / f'pca_cognitive_loadings_{model_name}.csv'
    efa_scores_path = PCA_RESULTS_DIR / f'efa_cognitive_scores_{model_name}.csv'
    efa_loadings_path = PCA_RESULTS_DIR / f'efa_cognitive_loadings_{model_name}.csv'
    efa_variance_path = PCA_RESULTS_DIR / f'efa_cognitive_variance_{model_name}.csv'
    efa_eigenvalues_path = PCA_RESULTS_DIR / f'efa_cognitive_eigenvalues_{model_name}.csv'
    efa_diagnostics_path = PCA_RESULTS_DIR / f'efa_cognitive_diagnostics_{model_name}.csv'
    efa_measure_diagnostics_path = PCA_RESULTS_DIR / f'efa_cognitive_measure_diagnostics_{model_name}.csv'

    pca_score_export = df[['src_subject_id']].join(cognitive_output['scores'])
    pca_score_export.to_csv(pca_scores_path, index=False)
    cognitive_output['variance'].to_csv(pca_variance_path, index=False)
    cognitive_output['loadings'].to_csv(pca_loadings_path, index_label='measure')

    efa_score_export = df[['src_subject_id']].join(cognitive_efa_output['scores'])
    efa_score_export.to_csv(efa_scores_path, index=False)
    cognitive_efa_output['loadings'].to_csv(efa_loadings_path, index_label='measure')
    cognitive_efa_output['variance'].to_csv(efa_variance_path, index=False)
    cognitive_efa_output['eigenvalues'].to_csv(efa_eigenvalues_path, index=False)
    cognitive_efa_output['diagnostics'].to_csv(efa_diagnostics_path, index=False)
    cognitive_efa_output['measure_diagnostics'].join(
        cognitive_efa_output['kmo_per_item']
    ).to_csv(efa_measure_diagnostics_path, index_label='measure')

    pca_export_manifest_rows.extend([
        {
            'model_name': model_name,
            'artifact': 'pca_cognitive_scores',
            'path': str(pca_scores_path),
            'n_rows': len(pca_score_export),
            'n_components': len(cognitive_output['score_cols']),
        },
        {
            'model_name': model_name,
            'artifact': 'pca_cognitive_variance',
            'path': str(pca_variance_path),
            'n_rows': len(cognitive_output['variance']),
            'n_components': len(cognitive_output['score_cols']),
        },
        {
            'model_name': model_name,
            'artifact': 'pca_cognitive_loadings',
            'path': str(pca_loadings_path),
            'n_rows': len(cognitive_output['loadings']),
            'n_components': len(cognitive_output['score_cols']),
        },
        {
            'model_name': model_name,
            'artifact': 'efa_cognitive_scores',
            'path': str(efa_scores_path),
            'n_rows': len(efa_score_export),
            'n_components': len(cognitive_efa_output['score_cols']),
        },
        {
            'model_name': model_name,
            'artifact': 'efa_cognitive_loadings',
            'path': str(efa_loadings_path),
            'n_rows': len(cognitive_efa_output['loadings']),
            'n_components': len(cognitive_efa_output['score_cols']),
        },
        {
            'model_name': model_name,
            'artifact': 'efa_cognitive_variance',
            'path': str(efa_variance_path),
            'n_rows': len(cognitive_efa_output['variance']),
            'n_components': len(cognitive_efa_output['score_cols']),
        },
        {
            'model_name': model_name,
            'artifact': 'efa_cognitive_eigenvalues',
            'path': str(efa_eigenvalues_path),
            'n_rows': len(cognitive_efa_output['eigenvalues']),
            'n_components': len(cognitive_efa_output['score_cols']),
        },
        {
            'model_name': model_name,
            'artifact': 'efa_cognitive_diagnostics',
            'path': str(efa_diagnostics_path),
            'n_rows': len(cognitive_efa_output['diagnostics']),
            'n_components': len(cognitive_efa_output['score_cols']),
        },
        {
            'model_name': model_name,
            'artifact': 'efa_cognitive_measure_diagnostics',
            'path': str(efa_measure_diagnostics_path),
            'n_rows': len(cognitive_efa_output['measure_diagnostics']),
            'n_components': len(cognitive_efa_output['score_cols']),
        },
    ])

pca_export_manifest = pd.DataFrame(pca_export_manifest_rows)
pca_export_manifest_path = PCA_RESULTS_DIR / 'pca_cognitive_export_manifest.csv'
pca_export_manifest.to_csv(pca_export_manifest_path, index=False)

print(f'Saved task PCA/EFA exports to {PCA_RESULTS_DIR}')
print(f'Saved task PCA/EFA export manifest to {pca_export_manifest_path}')
display(pca_export_manifest)
